# Row Level Security (RLS) — `vstone_catalog.security`

**Group -> Brand access matrix:**

| Group | Brands visible |
|---|---|
| `admin_group` | All brands (unrestricted) |
| `premium_users` | `bmw`, `mercedes-benz`, `lexus` |
| `toyota_users` | `toyota` only |
| `honda_users` | `honda` only |
| *(others)* | No rows (deny by default) |


## Step 1 -- Inspect the base Gold table

In [0]:
-- ============================================================
-- STEP 1: Inspect the base Gold table we will secure
-- ============================================================
SELECT * FROM vstone_catalog.gold.agg_top_10_brands_by_spend
ORDER BY total_market_value_usd DESC;

## Step 2 -- Verify workspace groups

In [0]:
-- ============================================================
-- STEP 2: Verify your workspace groups exist
-- ============================================================
SHOW GROUPS;

## Step 3 -- Confirm user identity and membership

In [0]:
-- ============================================================
-- STEP 3: Confirm current user identity and group membership
-- ============================================================
SELECT
  CURRENT_USER()                         AS current_user,
  is_account_group_member('toyota_users')              AS is_toyota_user,
  is_account_group_member('honda_users')               AS is_honda_user,
  is_account_group_member('premium_users')             AS is_premium_user,
  is_account_group_member('admins') AS is_admin;

## Step 4 -- Create RLS view in `security` schema

In [0]:
-- ============================================================
-- STEP 4: Create the RLS governance view
-- Location: vstone_catalog.security.rls_brand_market_data
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_brand_market_data AS
SELECT
  brand,
  total_market_value_usd,
  total_listings,
  avg_price_usd,
  gold_load_dt
FROM vstone_catalog.gold.agg_top_10_brands_by_spend
WHERE
  -- CASE evaluated per-row per-user; IS_MEMBER() resolves at query time
  CASE
    -- Admin sees everything
    WHEN is_account_group_member('admins')
      THEN TRUE

    -- Premium segment: BMW, Mercedes-Benz, Lexus
    WHEN is_account_group_member('premium_users')
      THEN brand IN ('bmw', 'mercedes-benz', 'lexus')

    -- Brand-specific groups
    WHEN is_account_group_member('toyota_users')
      THEN brand = 'toyota'

    WHEN is_account_group_member('honda_users')
      THEN brand = 'honda'

    -- Deny all unrecognised users
    ELSE FALSE
  END;

## Step 5 -- RLS on `fact_listings` (transaction grain)

In [0]:
-- ============================================================
-- STEP 5: Also apply RLS on fact_listings for row-level brand
-- fact_listings has no brand/model/fuel_type/price_category/
-- transmission_type/location_key columns (removed per schema update).
-- Resolved via dim joins; original RLS logic preserved exactly.
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_fact_listings AS
SELECT
  f.listing_id,
  dc.brand,              -- dim_car via car_sk
  dc.model,              -- dim_car via car_sk
  f.manufacture_year,
  f.listing_date,
  f.price_rub,
  f.price_usd,
  dp.price_category,     -- dim_price_category via price_category_key
  f.mileage_km,
  dc.fuel_type,          -- dim_car via car_sk
  dc.transmission,       -- dim_car via car_sk (was transmission_type)
  dl.city_name,          -- dim_location via location_sk (was location_key)
  f.photo_count,
  f.car_age_at_listing,
  f.is_high_mileage,
  f.price_per_hp_usd,
  f.gold_load_dt
FROM vstone_catalog.gold.fact_listings f
LEFT JOIN vstone_catalog.gold.dim_car            dc ON f.car_sk             = dc.car_sk
                                                   AND dc.__END_AT IS NULL
LEFT JOIN vstone_catalog.gold.dim_price_category dp ON f.price_category_key = dp.price_category_key
LEFT JOIN vstone_catalog.gold.dim_location       dl ON f.location_sk        = dl.location_sk
                                                   AND dl.__END_AT IS NULL
WHERE
  CASE
    WHEN is_account_group_member('admin_group') THEN TRUE
    WHEN is_account_group_member('premium_users')  THEN LOWER(TRIM(dc.brand)) IN ('bmw', 'mercedes-benz', 'lexus')
    WHEN is_account_group_member('toyota_users')   THEN LOWER(TRIM(dc.brand)) = 'toyota'
    WHEN is_account_group_member('honda_users')    THEN LOWER(TRIM(dc.brand)) = 'honda'
    ELSE FALSE
  END;

-- Verify
SELECT brand, COUNT(*) AS row_count
FROM vstone_catalog.security.rls_fact_listings
GROUP BY brand
ORDER BY row_count DESC;